# Customer features

Turn the existing 20,000-customer sample into one row per customer at a prediction date.

- Reuse the EDA definitions through `churn_retention/features.py`.
- Inspect a tiny synthetic example before building the real table.
- Preserve customers with absent history and keep supplied labels separate.

This notebook builds a **feature prototype**. It does not fit a model or report predictive performance.

## 1. What does 1 March mean?

Imagine we pause the clock at the **start of 1 March 2017**. We want to describe each customer using what we knew by then, so a future model could predict what happens next.

```text
PAST                             PRETEND WE PREDICT HERE       FUTURE
records through 28 February  -->  1 March, 00:00           -->  later outcomes
use these to build features       prediction date              labels must describe these
```

**1 March is our pretend “today”. It is not the date every customer churns.**

- A listening event or transaction dated **28 February** can contribute to features.
- An event dated **1 March or later** cannot contribute: it has not happened at the start of that day.
- A **lookback window** says how far into the past a particular feature looks. We still retain all available earlier listening history; we are not restricting the whole project to one month.

**Why did we choose 1 March?** Our retained original listening history ends on 28 February. Using the next morning lets us reproduce the existing EDA comparisons. This is a working example, not a date we have established as correct for modelling.

**Where do churn labels fit?** `train_v2.csv` supplies the answer, `is_churn`, which should describe what happened after a valid prediction moment. We still need to confirm its exact outcome period and customer eligibility. Until then, we keep those labels separate and mark the output **not ready for training**.

For a future historical training/evaluation split, we must also wait until each training customer's renewal-observation window has finished before treating their label as known.

## 2. Setup and source files

Run locally from this repository with its `.venv` kernel. Install `python -m pip install -e ".[notebooks]"` if needed. Run notebook 01 first if the listening sample is missing.

| Input | Use |
|---|---|
| `train_v2.csv` | Reconstruct the sampled customer IDs; hold supplied outcomes separately |
| Cached rows from original `user_logs.csv` | Earlier listening history for those IDs |
| Original `transactions.csv` or `transactions.csv.7z` | Earlier subscription events for those IDs |

No `transactions_v2.csv`, `user_logs_v2.csv` or profile fields enter this prototype.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from churn_retention.features import (
    FEATURE_COLUMNS,
    build_features,
    source_date_ranges,
    transaction_chunks,
)

RAW = ROOT / "data" / "raw"
OUTPUT = ROOT / "data" / "processed"
CACHE = OUTPUT / "cohort_cache"
PREDICTION_DATE = pd.Timestamp("2017-03-01")
SAMPLE_SIZE = 20_000
SEED = 42
print("Runtime: local repository; prediction date:", PREDICTION_DATE.date())

### See the dates used by the features

The table below calculates the calendar dates from `PREDICTION_DATE`. With the current 1 March setting, “last 7 days” means **22–28 February**, not 1–7 March. “Last 28 days” means **1–28 February**.

The 90-day view looks further back. The personal-baseline ratio compares the latest 28 days with the preceding 62 days; both periods are before the prediction date.

In [ ]:
window_dates = pd.DataFrame([
    {
        "History window": f"Last {days} days",
        "From (included)": (PREDICTION_DATE - pd.Timedelta(days=days)).date(),
        "Through (included)": (PREDICTION_DATE - pd.Timedelta(days=1)).date(),
    }
    for days in (7, 28, 90)
])
display(window_dates)
print("Personal baseline: recent 28 days versus the preceding 62 days.")

## 3. Reconstruct the same customer sample

Select IDs without looking at `is_churn`. The label table is checked for unique IDs and binary outcomes, but is not passed to `build_features`.

The cache manifest records a hash of the selected IDs. Matching that hash prevents silently using a different customer sample. The source size and modification time also identify the original listening input. A different source needs a regenerated cache.

In [ ]:
label_paths = list(RAW.rglob("train_v2.csv"))
if not label_paths:
    label_paths = list((OUTPUT / "cache").rglob("train_v2.csv"))
assert len(label_paths) == 1, "Keep one extracted train_v2.csv in data/raw, or run notebook 01."
labels = pd.read_csv(label_paths[0], dtype={"msno": "string", "is_churn": "Int8"})
assert labels["msno"].notna().all() and labels["msno"].is_unique
assert labels["is_churn"].isin([0, 1]).all()
cohort_ids = labels[["msno"]].sample(n=min(SAMPLE_SIZE, len(labels)), random_state=SEED)
cohort_ids = cohort_ids.sort_values("msno").reset_index(drop=True)

manifest = json.loads((CACHE / "precutoff_listening_manifest.json").read_text())
fingerprint = manifest["fingerprint"]
assert fingerprint["version"] == 2 and fingerprint["history_scope"] == "all_pre_cutoff"
assert fingerprint["seed"] == SEED and fingerprint["sample_size"] == len(cohort_ids)
ids_hash = hashlib.sha256("\n".join(cohort_ids["msno"]).encode()).hexdigest()
assert ids_hash == fingerprint["sample_ids_sha256"], "Sample IDs differ; regenerate in notebook 01."
assert PREDICTION_DATE <= pd.Timestamp(fingerprint["cutoff"]), "Cache lacks later history."
source_name = Path(fingerprint["source"]).name
assert source_name in {"user_logs.csv", "user_logs.csv.7z"}
listening_source = RAW / source_name
assert listening_source.stat().st_size == fingerprint["source_size"]
assert listening_source.stat().st_mtime_ns == fingerprint["source_mtime_ns"]
print("Verified sample customers:", len(cohort_ids))

## 4. A synthetic example

The fictional customer listens for 1 minute on 22 February and 2 minutes on 28 February. At the start of 1 March, that gives **2 recorded active days**, **3 recorded minutes** and **1 day since the last positive log** in the seven-day window.

The other fictional customer has no records. Missing duration/recency stays missing, and the coverage category explains why. Zero recorded active days does not prove zero real listening.

In [ ]:
synthetic_history = pd.DataFrame([
    ("synthetic-a", "20170222", 60),
    ("synthetic-a", "20170228", 120),
], columns=["msno", "date", "total_secs"])
synthetic_transactions = pd.DataFrame([
    ("synthetic-a", "20170227", 1, 0),
], columns=["msno", "transaction_date", "is_auto_renew", "is_cancel"])
synthetic_features, _ = build_features(
    synthetic_history, [synthetic_transactions],
    ["synthetic-a", "synthetic-no-history"], PREDICTION_DATE,
)
display(synthetic_features[[
    "customer_id", "recorded_active_days_7d", "recorded_minutes_28d",
    "days_since_last_positive_log", "history_coverage",
]])

## 5. Load the retained listening history

This reads the existing ~309 MB sample, not the 30 GB original CSV. The sample contains only ID, date and duration. Check its row count and customer membership before passing it to the feature function.

The function excludes events on or after the requested date, invalid durations and all duplicate customer/day rows. Quality counts remain visible below.

In [ ]:
history = pd.read_csv(
    CACHE / "precutoff_listening_sample.csv",
    dtype={"msno": "string", "date": "string", "total_secs": "float64"},
)
assert len(history) == manifest["selected_rows"]
assert history["msno"].isin(cohort_ids["msno"]).all()
cached_dates = pd.to_datetime(history["date"], format="%Y%m%d", errors="coerce")
assert cached_dates.dropna().lt(pd.Timestamp(fingerprint["cutoff"])).all()
print("Cached listening rows:", len(history))
print("Sample customers with listening rows:", history["msno"].nunique())

### 5.1. Raw inputs before feature calculation

Show the first five rows of each input again:

- **Labels:** rows from `train_v2.csv`.
- **Listening:** unsummarised `msno`, `date` and `total_secs` values retained from original `user_logs.csv` for the sampled customers. The other listening columns were not retained in this cache.
- **Transactions:** rows from original `transactions.csv`, including its payment and expiry fields. Read only the opening batch of the archive, without a full extraction.

These are separate table previews, not joined rows for the same five customers or representative samples. IDs are abbreviated only in the displayed copies. No cleaning or aggregation is applied to these previews.

**Reading the dates:** `20170228` means **28 February 2017**. Listening `date` and `transaction_date` say when an event happened. `membership_expire_date` says when a recorded subscription is due to end. A transaction on 20 February can already record a future expiry on 20 March; the event's transaction date determines whether it precedes our prediction point.

In [ ]:
from contextlib import closing

transaction_paths = [RAW / "transactions.csv", RAW / "transactions.csv.7z"]
transaction_path = next((p for p in transaction_paths if p.is_file()), None)
assert transaction_path is not None, "Download original transactions.csv.7z into data/raw."
with closing(transaction_chunks(transaction_path, chunksize=5, all_columns=True)) as chunks:
    raw_transactions_preview = next(chunks)

raw_previews = {
    "Labels — train_v2.csv": labels.head(),
    "Listening — cached rows from original user_logs.csv": history.head(),
    "Transactions — original transactions.csv": raw_transactions_preview,
}
for name, table in raw_previews.items():
    preview = table.copy()
    preview["msno"] = preview["msno"].str[:6] + "…"
    print(name)
    display(preview)

### 5.2. Earliest and latest dates: full files versus our sample

**The original downloads are unchanged.** Notebook 01 selected 20,000 customer IDs, then retained their original listening rows **before 1 March 2017**. That creates a separate saved sample. March rows in `user_logs_v2.csv` are not part of this sample.

**There is no lower date cutoff in the saved sample.** It keeps all available older rows for those IDs. The 7/28/90-day windows summarise different parts of that retained history; they do not delete older rows from the sample.

The first table below reads **all rows of each source's date columns**, without filtering by customer or prediction date. It then adds the actual range of the saved listening sample for comparison. The label files have no date columns: their filenames do not tell us their outcome periods. This audit checks coverage only; it does not add updated histories or profile fields to the feature table.

- `transaction_date` and listening `date`: when events happened.
- `membership_expire_date`: the recorded subscription expiry. A far-future expiry is not evidence that we have future transactions.
- `registration_init_time`: account registration dates, not listening history.

The original listening source has hundreds of millions of rows. Its first full check can take 10–20 minutes; the reader prints progress and does not extract the full CSV. Results are saved in the ignored `data/processed/source_date_ranges.json`. Later runs reuse them when the source file's path, size and modification time match. Invalid dates are counted and excluded from the reported minimum and maximum.

In [ ]:
source_date_columns = {
    "train.csv": [],
    "train_v2.csv": [],
    "members_v3.csv": ["registration_init_time"],
    "transactions.csv": ["transaction_date", "membership_expire_date"],
    "transactions_v2.csv": ["transaction_date", "membership_expire_date"],
    "user_logs.csv": ["date"],
    "user_logs_v2.csv": ["date"],
}
date_rows = []
for filename, date_columns in source_date_columns.items():
    source = RAW / filename
    if not source.is_file():
        source = RAW / (filename + ".7z")
    date_rows.extend(source_date_ranges(source, date_columns, OUTPUT / "source_date_ranges.json"))

sample_dates = pd.to_datetime(history["date"], format="%Y%m%d", errors="coerce")
date_rows.append({
    "file": "Saved listening sample", "date_column": "date",
    "earliest": sample_dates.min().date().isoformat(),
    "latest": sample_dates.max().date().isoformat(), "rows_checked": len(history),
    "invalid_dates": int(sample_dates.isna().sum()),
    "scope": "Selected customers; original logs before 1 March 2017",
})
source_date_table = pd.DataFrame(date_rows)
display(source_date_table[[
    "file", "date_column", "earliest", "latest", "scope",
]].fillna("No date column"))
print("The source-file ranges above are measured without our sample's date filter.")
original_last_date = source_date_table.loc[
    source_date_table["file"].eq("user_logs.csv"), "latest"
].iloc[0]
if pd.Timestamp(original_last_date) < pd.Timestamp(fingerprint["cutoff"]):
    print(f"Original user_logs.csv already ends on {original_last_date}.")
    print("The sample's date filter therefore removed no later dates from that original file.")

## 6. Build listening and subscription features

`build_features` produces these candidate groups:

| Group | Measurements |
|---|---|
| Subscription events | Latest earlier auto-renew flag, 28-day transaction/cancellation counts, transaction recency |
| Listening | Active days over 7/28/90 days, 28-day recorded minutes, positive-log recency |
| Personal change | Recent 28-day minutes per day divided by the preceding 62-day rate |
| Coverage and quality | Recent/older/absent source history, observed span, valid days and baseline eligibility |

The baseline ratio needs sufficiently early observation, no unusable rows in the 90-day period and a positive baseline. Undefined ratios stay missing. Latest-day conflicting renewal flags remain ambiguous; an event flag is not verified subscription state.

Read original transactions in bounded batches. A compressed input is streamed without creating another full extracted CSV. This scans the transaction source on each run and may take a few minutes; it does not rescan the original listening archive.

In [ ]:
print("Transaction source:", transaction_path.name)
features, quality = build_features(
    history, transaction_chunks(transaction_path), cohort_ids["msno"], PREDICTION_DATE,
)
print("Feature table shape:", features.shape)
preview = features.head().copy()
preview["customer_id"] = preview["customer_id"].str[:6] + "…"
display(preview)
display(pd.Series(quality["listening"], name="Listening quality"))
display(pd.Series(quality["transactions"], name="Transaction quality"))

## 7. Check keys, bounds and missing values

Every sampled customer must survive the joins, including those without listening or transactions. Customer IDs and prediction dates are keys, not predictors. `FEATURE_COLUMNS` is the explicit predictor list; `is_churn` is absent.

Do not fill unknown minutes, recency or baseline ratios with zero. Their missingness and coverage flags need an explicit preprocessing policy when modelling begins. Synthetic tests check cutoff boundaries, future-record invariance, conflicting flags and invalid rows.

In [ ]:
assert len(features) == len(cohort_ids)
assert features["customer_id"].is_unique
assert set(features["customer_id"]) == set(cohort_ids["msno"])
assert features.columns.tolist() == ["customer_id", "prediction_date", *FEATURE_COLUMNS]
assert "is_churn" not in features.columns
for days in (7, 28, 90):
    assert features[f"recorded_active_days_{days}d"].between(0, days).all()
assert not features.select_dtypes("number").isin([float("inf"), float("-inf")]).any().any()
display(features[list(FEATURE_COLUMNS)].isna().sum().rename("Missing customers").to_frame())
display(features["history_coverage"].value_counts().rename("Customers").to_frame())

## 8. Keep supplied labels separate

Preserve the sampled source labels in their own table. We have not assigned a verified prediction/outcome period or label-availability date to these labels. Do not pair them with the candidate feature date for training until the timeline is established.

In [ ]:
supplied_labels = cohort_ids.merge(labels, on="msno", validate="one_to_one")
supplied_labels = supplied_labels.rename(columns={"msno": "customer_id"})
supplied_labels["label_source"] = "train_v2.csv"
assert len(supplied_labels) == len(features)
print("Supplied labels held separately:", len(supplied_labels))

## 9. Save the prototype

Save three ignored files directly under `data/processed/`:

- `customer_features.csv`: one customer/date row with 15 candidate predictors; missing values use empty CSV cells.
- `supplied_labels_sample.csv`: unchanged supplied outcomes and their source, without an invented outcome date.
- `feature_metadata.json`: exact inputs, sample hash, cutoff, windows, quality counts and unresolved timing checks.

The next step is to verify the label periods and prediction-time population, then design mature training/evaluation cohorts. No random split or model is created here.

In [ ]:
metadata = {
    "stage": "feature_prototype", "ready_for_training": False,
    "label_period_verified": False, "eligible_population_verified": False,
    "ingestion_availability_verified": False, "label_available_date": None,
    "prediction_date": PREDICTION_DATE.date().isoformat(),
    "sample_size": len(features), "seed": SEED, "sample_ids_sha256": ids_hash,
    "sources": {
        "customer_ids_and_supplied_labels": str(label_paths[0].relative_to(ROOT)),
        "listening_history": str((CACHE / "precutoff_listening_sample.csv").relative_to(ROOT)),
        "listening_source_fingerprint": fingerprint,
        "transactions": str(transaction_path.relative_to(ROOT)),
        "transaction_source_size": transaction_path.stat().st_size,
        "transaction_source_mtime_ns": transaction_path.stat().st_mtime_ns,
    },
    "feature_columns": list(FEATURE_COLUMNS), "lookback_days": [7, 28, 90],
    "personal_baseline_days": {"recent": 28, "earlier": 62},
    "quality": quality, "pandas_version": pd.__version__,
    "models_trained": False,
}
OUTPUT.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT / "customer_features.csv", index=False)
supplied_labels.to_csv(OUTPUT / "supplied_labels_sample.csv", index=False)
(OUTPUT / "feature_metadata.json").write_text(json.dumps(metadata, indent=2, allow_nan=False))
print("Saved prototype outputs to:", OUTPUT)
print("Label alignment remains unverified. This prototype is not ready for model training.")